# Training a YOLO model and recording its lineage in FiftyOne

This notebook runs the **real** thing: it fine-tunes an Ultralytics YOLO model on `quickstart`, then records that training as a **lineage record** on the dataset.

The record is **one object** answering four questions:

- **What data** did the model see? → the frozen `train` / `val` / `test` views
- **Where is the model?** → `checkpoint_uri`
- **Where are the experiment notes?** → `project_url`
- **How good was it?** → a linked FiftyOne **evaluation**

End to end: split → export → train → `init_training_run` → `apply_model` → `finish` (auto-eval + weld) → inspect → review → manage → and finally the **Training Runs** panel.

## 1. Setup

In [ ]:
!pip install -q fiftyone ultralytics

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.types as fot
import fiftyone.utils.random as four
from ultralytics import YOLO

## 2. Build train / val / test splits

Clone `quickstart` to a throwaway dataset, then tag a 70/15/15 split. The training API freezes each split's membership to sample IDs at `init` time, so these tags define exactly what each split contains.

In [ ]:
NAME = "yolo-lineage-demo"
if fo.dataset_exists(NAME):
    fo.delete_dataset(NAME)

dataset = foz.load_zoo_dataset("quickstart").clone(name=NAME)
dataset.persistent = True
four.random_split(dataset, {"train": 0.7, "val": 0.15, "test": 0.15}, seed=51)

for tag in ("train", "val", "test"):
    print("%-5s %d" % (tag, dataset.match_tags(tag).count()))

## 3. Export the splits to YOLO format

FiftyOne writes a `dataset.yaml` that Ultralytics can train on directly.

In [ ]:
EXPORT_DIR = "/tmp/yolo_lineage_export"
classes = dataset.distinct("ground_truth.detections.label")

for split in ("train", "val"):
    dataset.match_tags(split).export(
        export_dir=EXPORT_DIR,
        dataset_type=fot.YOLOv5Dataset,
        label_field="ground_truth",
        split=split,
        classes=classes,
    )

print("wrote", EXPORT_DIR + "/dataset.yaml", "with", len(classes), "classes")

## 4. Fine-tune a real YOLO model

A few epochs on `yolov8n` is enough to produce a genuine checkpoint and real predictions.

In [ ]:
model = YOLO("yolov8n.pt")
model.train(
    data=EXPORT_DIR + "/dataset.yaml",
    epochs=10,
    imgsz=640,
    project="/tmp/yolo_lineage_runs",
    name="quickstart",
)
checkpoint = str(model.trainer.best)   # path to best.pt
print("best checkpoint:", checkpoint)

## 5. Open the lineage record — `init_training_run`

Snapshots the three views to frozen sample IDs and registers the run with `status='in_progress'`. `train_key` must be a valid identifier. Because a `test_view` is present, `auto_eval` defaults to `True`.

In [ ]:
TRAIN_KEY = "yolov8n_quickstart"

run = dataset.init_training_run(
    train_key=TRAIN_KEY,
    train_view=dataset.match_tags("train"),
    val_view=dataset.match_tags("val"),
    test_view=dataset.match_tags("test"),
    gt_field="ground_truth",
    pred_field="yolo_predictions",
    auto_eval=True,
    project_url="https://wandb.ai/me/yolov8n-quickstart",
    train_config={"model": "yolov8n.pt", "epochs": 10, "imgsz": 640},
)

print("status :", run.status)
print("splits : train=%d val=%d test=%d" % (
    run.train_view.count(), run.val_view.count(), run.test_view.count()))

## 6. Run the trained model on the test split — `run.apply_model`

FiftyOne applies the Ultralytics model natively, writing `Detections` into `yolo_predictions` on the frozen test view.

In [ ]:
run.apply_model(model, samples=run.test_view)

print("test samples with predictions:",
      run.test_view.exists("yolo_predictions").count())

## 7. `finish` — auto-eval and weld the evaluation

`finish` evaluates the populated predictions against ground truth, links that evaluation to the run (`eval_key` defaults to `train_key`), and stamps `status='completed'`. `checkpoint_uri` records *where* the weights live — FiftyOne never stores the bytes.

In [ ]:
run.finish(checkpoint_uri=checkpoint)

print("status         :", run.status)
print("eval_key       :", run.eval_key)
print("checkpoint_uri :", run.checkpoint_uri)
print("evaluated      :", run.eval_view.count(), "samples")

## 8. The four answers + the eval weld (both directions)

One record, four answers — and the weld that makes this a FiftyOne feature: forward `run → eval`, back `eval → train_key`.

In [ ]:
info = dataset.get_training_info(TRAIN_KEY).config
print("what  (data)  : train=%d val=%d test=%d frozen ids" % (
    len(info.train_view_ids), len(info.val_view_ids), len(info.test_view_ids)))
print("where (model) :", info.checkpoint_uri)
print("notes (exp)   :", info.project_url)
print("how good      :", run.eval_results.metrics())

# the evaluation knows which run produced it
einfo = dataset.get_evaluation_info(run.eval_key)
print("eval -> train_key:", einfo.config.train_key)

## 9. Review lifecycle — `review_status` + `note`

The human-review fields the Training Runs panel reads/writes, kept distinct from the execution `status`. Set them from the SDK and they round-trip on the record.

In [ ]:
run.config.review_status = "candidate"
run.config.note = "yolov8n, 10 epochs - promising baseline"
run.save_config()

c = dataset.get_training_info(TRAIN_KEY).config
print("review_status :", c.review_status)
print("note          :", c.note)
print("exec status   :", c.status)   # unchanged: 'completed'

## 10. Manage runs — list / get / load

In [ ]:
print("runs           :", dataset.list_training_runs())
print("completed runs :", dataset.list_training_runs(status="completed"))

reloaded = dataset.load_training_run(TRAIN_KEY)
print("reloaded status:", reloaded.status, "| eval:", reloaded.eval_key)
print("reloaded views : train=%d val=%d test=%d" % (
    reloaded.train_view.count(), reloaded.val_view.count(), reloaded.test_view.count()))

## 11. Door 3 — log an externally-trained model in one call

`add_training_run` is `init` + `finish` in one call: record a model you trained elsewhere. With `auto_eval=False` it associates the lineage without running an evaluation (pass `eval_key=...` to weld an existing one).

In [ ]:
ext = dataset.add_training_run(
    "external_yolo_v2",
    train_view=dataset.match_tags("train"),
    test_view=dataset.match_tags("test"),
    gt_field="ground_truth",
    auto_eval=False,
    checkpoint_uri="s3://my-bucket/external/yolo_v2.pt",
    project_url="https://wandb.ai/me/external-yolo-v2",
    train_config={"model": "yolov8s.pt", "epochs": 50},
)
print("logged:", ext.train_key, "| status:", ext.status, "| eval:", ext.eval_key)
print("all runs:", dataset.list_training_runs())

## 12. See it in the Training Runs panel

Launch the App and open the **Training Runs** panel (grid panel menu). Each run is a card with its execution-status pill, training-set sample count, and review status; opening a card shows the **Training configuration** section (from `train_config`), the **eval** summary that deep-links the Model Evaluation panel, and the editable note.

In [ ]:
session = fo.launch_app(dataset)

## Cleanup (optional)

Run when you're done exploring the panel.

In [ ]:
# fo.delete_dataset(NAME)